# Uncertainty Quantification Analysis

This notebook provides an interactive analysis of the uncertainty quantification improvements implemented in the ML model.

## Contents

1. Setup and Data Loading
2. Basic Model Evaluation
3. Calibration Analysis (ECE & Reliability Diagram)
4. Multi-Metric Uncertainty Analysis
5. Uncertainty Separation Analysis
6. Threshold Optimization
7. Temperature Scaling Calibration
8. Recommendations

## 1. Setup and Data Loading

In [ ]:
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Import uncertainty metrics
from app.services.ml_service import (
    eng_classifier,
    fil_classifier,
    compute_expected_calibration_error,
    compute_multi_metric_uncertainty,
    compute_reliability_diagram_data,
    compute_uncertainty_separation,
    optimize_uncertainty_threshold,
    apply_temperature_scaling,
    find_optimal_temperature,
)

# Plotting configuration
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

print("✓ Libraries loaded successfully")

In [ ]:
# Load test dataset
def load_test_dataset(dataset_path: str):
    with open(dataset_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    symptoms_list = [item["symptoms"] for item in data]
    true_labels = [item["disease"] for item in data]
    
    return symptoms_list, true_labels

# Load the sample test data
symptoms_list, true_labels = load_test_dataset("scripts/test_data.json")
print(f"Loaded {len(symptoms_list)} test samples")
print(f"Diseases: {set(true_labels)}")

## 2. Basic Model Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def evaluate_classifier(classifier, symptoms_list, true_labels, model_name="Model"):
    """Run predictions and collect all metrics"""
    predictions = []
    confidences = []
    all_mc_predictions = []
    all_mean_probs = []
    all_std_probs = []
    all_logits = []
    
    for symptoms in symptoms_list:
        result = classifier.predict_with_uncertainty(symptoms)
        
        predictions.append(result["predicted_label"][0])
        confidences.append(float(result["confidence"][0]))
        all_mc_predictions.append(result["mean_probabilities"])
        all_mean_probs.append(result["mean_probabilities"][0])
        all_std_probs.append(result["std_probabilities"][0])
    
    # Convert to numpy
    predictions = np.array(predictions)
    confidences = np.array(confidences)
    all_mc_predictions = np.array(all_mc_predictions)
    all_mean_probs = np.array(all_mean_probs)
    all_std_probs = np.array(all_std_probs)
    
    # Convert labels to indices
    label_to_idx = classifier.model.config.label2id
    true_labels_idx = np.array([label_to_idx[label] for label in true_labels])
    pred_labels_idx = np.array([label_to_idx[pred] for pred in predictions])
    
    # Compute accuracy
    accuracies = (pred_labels_idx == true_labels_idx).astype(int)
    accuracy = accuracy_score(true_labels_idx, pred_labels_idx)
    
    print(f"\n{'='*60}")
    print(f"{model_name}")
    print(f"{'='*60}")
    print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"Average Confidence: {np.mean(confidences):.4f}")
    print(f"Calibration Gap: {np.mean(confidences) - accuracy:.4f}")
    
    return {
        "predictions": predictions,
        "confidences": confidences,
        "mc_predictions": all_mc_predictions,
        "mean_probs": all_mean_probs,
        "std_probs": all_std_probs,
        "true_labels_idx": true_labels_idx,
        "pred_labels_idx": pred_labels_idx,
        "accuracies": accuracies,
        "accuracy": accuracy,
    }

# Evaluate English model
eng_results = evaluate_classifier(eng_classifier, symptoms_list, true_labels, "BioClinical ModernBERT (English)")

## 3. Calibration Analysis

In [ ]:
# Compute Expected Calibration Error
ece = compute_expected_calibration_error(
    eng_results["pred_labels_idx"],
    eng_results["confidences"],
    eng_results["true_labels_idx"],
    n_bins=10
)

print(f"Expected Calibration Error (ECE): {ece:.4f}")
print(f"Interpretation: {'Well calibrated' if ece < 0.05 else 'Needs calibration'}")

In [ ]:
# Plot Reliability Diagram
rel_data = compute_reliability_diagram_data(
    eng_results["confidences"],
    eng_results["accuracies"],
    n_bins=10
)

fig, ax = plt.subplots(figsize=(10, 8))

# Perfect calibration line
ax.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration', linewidth=2)

# Actual calibration
ax.bar(rel_data["bin_centers"], rel_data["accuracies"], width=0.08, 
       alpha=0.7, label='Actual Accuracy', edgecolor='black')

ax.set_xlabel('Mean Predicted Confidence', fontsize=14)
ax.set_ylabel('Fraction of Correct Predictions', fontsize=14)
ax.set_title('Reliability Diagram (Calibration Curve)', fontsize=16)
ax.legend(loc='upper left')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3)

# Add ECE annotation
ax.text(0.95, 0.05, f'ECE = {ece:.4f}', transform=ax.transAxes, 
        fontsize=12, verticalalignment='bottom', horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## 4. Multi-Metric Uncertainty Analysis

In [ ]:
# Compute multi-metric uncertainty for all predictions
all_uncertainties = []
for i in range(len(eng_results["mc_predictions"])):
    unc_metrics = compute_multi_metric_uncertainty(
        eng_results["mc_predictions"][i],
        eng_results["mean_probs"][i],
        eng_results["std_probs"][i]
    )
    all_uncertainties.append(unc_metrics)

# Extract individual metrics
mi_values = np.array([u["mutual_information"] for u in all_uncertainties])
entropy_values = np.array([u["predictive_entropy"] for u in all_uncertainties])
variance_values = np.array([u["variance"] for u in all_uncertainties])
cv_values = np.array([u["coefficient_of_variation"] for u in all_uncertainties])
disagreement_values = np.array([u["ensemble_disagreement"] for u in all_uncertainties])

# Plot uncertainty metric distributions
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

metrics = [
    (mi_values, "Mutual Information", "blue"),
    (entropy_values, "Predictive Entropy", "green"),
    (variance_values, "Variance", "red"),
    (cv_values, "Coefficient of Variation", "purple"),
    (disagreement_values, "Ensemble Disagreement", "orange"),
    (eng_results["confidences"], "Confidence", "teal"),
]

for idx, (values, name, color) in enumerate(metrics):
    axes[idx].hist(values, bins=20, color=color, alpha=0.7, edgecolor='black')
    axes[idx].axvline(np.mean(values), color='black', linestyle='--', 
                     label=f'Mean: {np.mean(values):.4f}')
    axes[idx].set_xlabel(name)
    axes[idx].set_ylabel('Frequency')
    axes[idx].set_title(f'{name} Distribution')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print statistics
print("\nUncertainty Metric Statistics:")
print(f"  Mutual Information:     {mi_values.mean():.4f} ± {mi_values.std():.4f}")
print(f"  Predictive Entropy:     {entropy_values.mean():.4f} ± {entropy_values.std():.4f}")
print(f"  Variance:               {variance_values.mean():.4f} ± {variance_values.std():.4f}")
print(f"  Coefficient of Variation: {cv_values.mean():.4f} ± {cv_values.std():.4f}")
print(f"  Ensemble Disagreement:  {disagreement_values.mean():.4f} ± {disagreement_values.std():.4f}")

## 5. Uncertainty Separation Analysis

In [ ]:
# Analyze how well each metric separates correct from incorrect predictions
print("="*60)
print("UNCERTAINTY SEPARATION ANALYSIS")
print("="*60)

metrics_to_analyze = [
    (mi_values, "Mutual Information"),
    (variance_values, "Variance"),
    (cv_values, "Coefficient of Variation"),
    (disagreement_values, "Ensemble Disagreement"),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, (values, name) in enumerate(metrics_to_analyze):
    separation = compute_uncertainty_separation(values, eng_results["accuracies"])
    
    if separation["can_evaluate"]:
        print(f"\n{name}:")
        print(f"  Mean (Correct):   {separation['mean_uncertainty_correct']:.4f}")
        print(f"  Mean (Incorrect): {separation['mean_uncertainty_incorrect']:.4f}")
        print(f"  Separation:       {separation['separation_mean']:.4f}")
        print(f"  AUC-ROC:          {separation['auc']:.4f}")
        
        # Plot distribution
        correct_vals = values[eng_results["accuracies"] == 1]
        incorrect_vals = values[eng_results["accuracies"] == 0]
        
        axes[idx].hist(correct_vals, bins=15, alpha=0.6, label='Correct', color='green')
        axes[idx].hist(incorrect_vals, bins=15, alpha=0.6, label='Incorrect', color='red')
        axes[idx].axvline(separation['mean_uncertainty_correct'], color='green', 
                         linestyle='--', linewidth=2)
        axes[idx].axvline(separation['mean_uncertainty_incorrect'], color='red', 
                         linestyle='--', linewidth=2)
        axes[idx].set_xlabel(name)
        axes[idx].set_ylabel('Frequency')
        axes[idx].set_title(f'{name}: Correct vs Incorrect (AUC={separation["auc"]:.3f})')
        axes[idx].legend()
        axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Threshold Optimization

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

print("="*60)
print("THRESHOLD OPTIMIZATION")
print("="*60)

# Optimize MI threshold
mi_optimal = optimize_uncertainty_threshold(mi_values, eng_results["accuracies"], metric="f1")

print(f"\nMutual Information Optimal Threshold:")
print(f"  F1-optimal:     {mi_optimal['threshold']:.4f}")
print(f"  ROC-optimal:    {mi_optimal['roc_optimal_threshold']:.4f}")
print(f"  PR-optimal:     {mi_optimal['pr_optimal_threshold']:.4f}")

# Plot ROC curve
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ROC Curve
fpr, tpr, thresholds = roc_curve(eng_results["accuracies"] == 0, mi_values)
youden_index = tpr - fpr
optimal_idx = np.argmax(youden_index)

axes[0].plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC Curve (AUC={np.trapz(tpr, fpr):.3f})')
axes[0].plot(fpr[optimal_idx], tpr[optimal_idx], 'ro', markersize=10, 
            label=f'Optimal (T={thresholds[optimal_idx]:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve for Uncertainty-Based Error Detection')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Precision-Recall Curve
precision, recall, pr_thresholds = precision_recall_curve(eng_results["accuracies"] == 0, mi_values)
pr_scores = 2 * precision * recall / (precision + recall + 1e-10)
optimal_pr_idx = np.argmax(pr_scores)

axes[1].plot(recall, precision, 'g-', linewidth=2, 
            label=f'PR Curve (F1={pr_scores[optimal_pr_idx]:.3f})')
axes[1].plot(recall[optimal_pr_idx], precision[optimal_pr_idx], 'ro', markersize=10,
            label=f'Optimal (T={pr_thresholds[optimal_pr_idx]:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Temperature Scaling Calibration

In [ ]:
# Find optimal temperature
# Note: We need logits, not probabilities. Let's approximate from mean_probs
# In practice, you'd modify the classifier to return logits

print("Temperature Scaling Analysis")
print("="*60)

# Convert probabilities to approximate logits (inverse softmax)
# This is an approximation; real implementation should use actual logits
def probs_to_logits(probs, epsilon=1e-10):
    probs = np.clip(probs, epsilon, 1 - epsilon)
    return np.log(probs)

approx_logits = np.array([probs_to_logits(p) for p in eng_results["mean_probs"]])

# Find optimal temperature
optimal_temp = find_optimal_temperature(approx_logits, eng_results["true_labels_idx"], 
                                        temperature_range=(0.5, 3.0), n_steps=25)

print(f"Optimal Temperature: {optimal_temp:.3f}")
print(f"Interpretation:")
if optimal_temp > 1.1:
    print(f"  Model is overconfident. Apply T={optimal_temp:.2f} to soften predictions.")
elif optimal_temp < 0.9:
    print(f"  Model is underconfident. Apply T={optimal_temp:.2f} to sharpen predictions.")
else:
    print(f"  Model is well-calibrated. No temperature scaling needed.")

# Plot temperature vs NLL
temperatures = np.linspace(0.5, 3.0, 25)
nll_values = []

for temp in temperatures:
    scaled_probs = apply_temperature_scaling(approx_logits, temp)
    nll = -np.mean(np.log(scaled_probs[np.arange(len(eng_results["true_labels_idx"])), 
                                       eng_results["true_labels_idx"]] + 1e-10))
    nll_values.append(nll)

plt.figure(figsize=(10, 6))
plt.plot(temperatures, nll_values, 'b-o', linewidth=2, markersize=6)
plt.axvline(optimal_temp, color='red', linestyle='--', 
           label=f'Optimal T={optimal_temp:.3f}')
plt.axvline(1.0, color='green', linestyle='--', alpha=0.7, label='No Scaling (T=1.0)')
plt.xlabel('Temperature')
plt.ylabel('Negative Log Likelihood (NLL)')
plt.title('Temperature Scaling Optimization')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 8. Summary and Recommendations

In [ ]:
print("="*60)
print("SUMMARY AND RECOMMENDATIONS")
print("="*60)

recommendations = []

# Calibration assessment
calibration_gap = np.mean(eng_results["confidences"]) - eng_results["accuracy"]
if abs(calibration_gap) > 0.05:
    if calibration_gap > 0:
        recommendations.append(f"⚠️  Model is OVERCONFIDENT (gap={calibration_gap:.3f})")
        recommendations.append(f"   → Apply temperature scaling with T ≈ {optimal_temp:.2f}")
    else:
        recommendations.append(f"⚠️  Model is UNDERCONFIDENT (gap={calibration_gap:.3f})")
        recommendations.append(f"   → Apply temperature scaling with T ≈ {optimal_temp:.2f}")
else:
    recommendations.append("✓ Model is well-calibrated")

# ECE assessment
if ece < 0.05:
    recommendations.append(f"✓ ECE is acceptable ({ece:.4f})")
else:
    recommendations.append(f"⚠️  ECE needs improvement ({ece:.4f} > 0.05)")

# Uncertainty separation assessment
mi_separation = compute_uncertainty_separation(mi_values, eng_results["accuracies"])
if mi_separation["can_evaluate"]:
    if mi_separation["auc"] > 0.7:
        recommendations.append(f"✓ MI effectively separates correct/incorrect (AUC={mi_separation['auc']:.3f})")
    elif mi_separation["auc"] > 0.6:
        recommendations.append(f"⚠️  MI has moderate separation (AUC={mi_separation['auc']:.3f})")
    else:
        recommendations.append(f"⚠️  MI has poor separation (AUC={mi_separation['auc']:.3f})")

# Threshold recommendations
recommendations.append(f"\nRecommended MI threshold: {mi_optimal['roc_optimal_threshold']:.4f}")
recommendations.append(f"Current threshold: 0.05")
recommendations.append(f"Consider multi-metric approach: MI + Variance + Ensemble Disagreement")

for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec}")

print("\n" + "="*60)